# 🔬 Reconstruct Balanced Master Dataset v6

**Goal:** Create a "World" that teaches the model logic, not noise.

**Strategy:**
1. Merge Train + Test (132M + 22M rows)
2. Clean toxic rows
3. Create 50/50 balanced pools

**Output:**
- `master_train_v6_balanced.parquet`
- `master_val_v6.parquet`

## 1. Setup

In [ ]:
!pip install polars -q

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl
import os

print(f"Polars: {pl.__version__}")

In [ ]:
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
RAW_PATH = f"{BASE_PATH}/raw"
OUTPUT_PATH = f"{BASE_PATH}/processed"

FILES = {
    "train": f"{RAW_PATH}/metabonet_public_train.parquet",
    "test": f"{RAW_PATH}/metabonet_public_test.parquet",
    "output_train": f"{OUTPUT_PATH}/master_train_v6_balanced.parquet",
    "output_val": f"{OUTPUT_PATH}/master_val_v6.parquet"
}

for name, path in FILES.items():
    if "output" not in name:
        exists = "✅" if os.path.exists(path) else "❌"
        print(f"{exists} {name}: {path}")

## 2. Clean & Scan (Lazy Mode)

In [ ]:
# Toxic mask: DROP these rows
def get_toxic_mask():
    return (
        (pl.col("CGM") < 110) & 
        (pl.col("carbs") == 0) & 
        (pl.col("bolus") > 0)
    )

# Columns to keep
COLS = ['CGM', 'bolus', 'carbs', 'basal', 'age', 'weight', 
        'time_sin', 'time_cos', 'cgm_delta_5min', 'cgm_delta_15min', 
        'is_weekend', 'gender_encoded', 'insulin_type_encoded']

print("✅ Toxic mask defined")

In [ ]:
print("📂 Scanning TRAIN...")
lf_train = pl.scan_parquet(FILES["train"])
train_cols = [c for c in COLS if c in lf_train.columns]
lf_train = lf_train.select(train_cols).filter(~get_toxic_mask())
print(f"   Columns: {len(train_cols)}")

print("\n📂 Scanning TEST...")
lf_test = pl.scan_parquet(FILES["test"])
test_cols = [c for c in COLS if c in lf_test.columns]
lf_test = lf_test.select(test_cols).filter(~get_toxic_mask())
print(f"   Columns: {len(test_cols)}")

# Common columns
common_cols = list(set(train_cols) & set(test_cols))
print(f"\n✅ Common columns: {common_cols}")

In [ ]:
# Concatenate (lazy)
print("🔗 Concatenating train + test (lazy)...")
lf_all = pl.concat([
    lf_train.select(common_cols),
    lf_test.select(common_cols)
])

# Count (streaming)
print("📊 Counting rows (streaming)...")
total_count = lf_all.select(pl.count()).collect(streaming=True).item()
print(f"\n✅ Total clean rows: {total_count:,}")

## 3. Create Balanced Pools

In [ ]:
# Pool Action: bolus > 0
print("🎯 Creating Pool_Action (bolus > 0)...")
pool_action = (
    lf_all
    .filter(pl.col("bolus") > 0)
    .collect(streaming=True)
)
n_action = len(pool_action)
print(f"   Pool_Action: {n_action:,} rows")

In [ ]:
# Pool Safe: bolus == 0
print("🛡️ Creating Pool_Safe (bolus == 0)...")
pool_safe_all = (
    lf_all
    .filter(pl.col("bolus") == 0)
    .collect(streaming=True)
)
n_safe_all = len(pool_safe_all)
print(f"   Total safe rows: {n_safe_all:,}")

In [ ]:
# Balance by CGM range
print("\n⚖️ Balancing Safe Pool by CGM range...")

target_safe = min(n_action, n_safe_all)

# Split by CGM
pool_safe_low = pool_safe_all.filter(pl.col("CGM") < 120)
pool_safe_high = pool_safe_all.filter(pl.col("CGM") >= 120)

n_low = len(pool_safe_low)
n_high = len(pool_safe_high)

print(f"   Low CGM (<120): {n_low:,}")
print(f"   High CGM (≥120): {n_high:,}")

# Sample 50% from each
target_low = target_safe // 2
target_high = target_safe - target_low

sample_low = min(target_low, n_low)
sample_high = min(target_high, n_high)

print(f"\n   Sampling: {sample_low:,} low + {sample_high:,} high")

In [ ]:
# Sample
print("🎲 Sampling...")

if sample_low > 0 and sample_low < n_low:
    pool_safe_low_sampled = pool_safe_low.sample(n=sample_low, seed=42)
else:
    pool_safe_low_sampled = pool_safe_low

if sample_high > 0 and sample_high < n_high:
    pool_safe_high_sampled = pool_safe_high.sample(n=sample_high, seed=42)
else:
    pool_safe_high_sampled = pool_safe_high

# Combine
pool_safe = pl.concat([pool_safe_low_sampled, pool_safe_high_sampled])
print(f"   Final Pool_Safe: {len(pool_safe):,} rows")

## 4. Merge & Shuffle

In [ ]:
print("🔀 Merging pools...")
df_master = pl.concat([pool_action, pool_safe])
print(f"   Combined: {len(df_master):,} rows")

print("\n🎲 Shuffling...")
df_master = df_master.sample(fraction=1.0, seed=42, shuffle=True)

print("\n📐 Casting to Float32...")
float_cols = [c for c in df_master.columns if df_master[c].dtype in [pl.Float64, pl.Int64]]
df_master = df_master.cast({c: pl.Float32 for c in float_cols})

print(f"\n✅ Master dataset ready: {len(df_master):,} rows")

## 5. Split & Export

In [ ]:
VAL_SIZE = 1_000_000

val_size = min(VAL_SIZE, len(df_master) // 10)
print(f"📊 Splitting validation: {val_size:,} rows")

df_val = df_master.head(val_size)
df_train = df_master.slice(val_size)

print(f"   Train: {len(df_train):,}")
print(f"   Val: {len(df_val):,}")

In [ ]:
print(f"💾 Saving train to: {FILES['output_train']}")
df_train.write_parquet(FILES['output_train'])
print("✅ Train saved!")

print(f"\n💾 Saving val to: {FILES['output_val']}")
df_val.write_parquet(FILES['output_val'])
print("✅ Val saved!")

## 6. Summary

In [ ]:
# Balance check
train_action = len(df_train.filter(pl.col("bolus") > 0))
train_safe = len(df_train.filter(pl.col("bolus") == 0))

print("\n" + "=" * 60)
print("📋 RECONSTRUCTION SUMMARY")
print("=" * 60)
print(f"""
OUTPUT FILES:
   ✅ {FILES['output_train']}
   ✅ {FILES['output_val']}

SIZES:
   Master Train: {len(df_train):,}
   Master Val:   {len(df_val):,}

BALANCE (Train):
   Action (bolus>0): {train_action:,} ({train_action/len(df_train)*100:.1f}%)
   Safe (bolus=0):   {train_safe:,} ({train_safe/len(df_train)*100:.1f}%)

CGM BALANCE (Safe Pool):
   Low CGM (<120):  {sample_low:,}
   High CGM (≥120): {sample_high:,}
""")
print("=" * 60)
print("🎉 DONE!")